## Planning:
- Load data
- Create Feature Engineering + No Dummify
- Model XGBoost, standard parameter, 5 cv

In [2]:
# data processing
import pandas as pd
import numpy as np

# data visualisation
import seaborn as sns
import matplotlib.pyplot as plt

# data modeling 
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# model 
from sklearn.linear_model import LogisticRegression
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, Pool

In [3]:
import gc

In [4]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
submit = pd.read_csv("sample_submission.csv")

In [5]:
train['Arrival Delay in Minutes'] = train['Arrival Delay in Minutes'].fillna(train['Departure Delay in Minutes'])

In [6]:
test['Arrival Delay in Minutes'] = test['Arrival Delay in Minutes'].fillna(test['Departure Delay in Minutes'])

In [7]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()

In [8]:
non_numeric_cols = train.select_dtypes(exclude='number').columns.tolist()

## Feature Engineering

In [9]:
# splitting X and X_test, take out satisfaction
X = train.drop(columns=["id","satisfaction"])
y = train["satisfaction"]
X_test = test.drop(columns=["id"])

Inspiration: https://www.kaggle.com/competitions/playground-series-s6e10/discussion/754317

In [10]:
svc = ['Inflight wifi service','Departure/Arrival time convenient','Ease of Online booking',
       'Gate location','Food and drink','Online boarding','Seat comfort','Inflight entertainment',
       'On-board service','Leg room service','Baggage handling','Checkin service','Cleanliness']

In [15]:
# menghitung rata-rata rating dan total rating pada setiap row
for df in (X, X_test):
    df['svc_sum'] = df[svc].sum(axis=1)
    df['svc_mean'] = df[svc].mean(axis=1)

In [16]:
X.head(2)

,Age,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,Gate location,Food and drink,Online boarding,Seat comfort,Inflight entertainment,...,Checkin service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,Gender,Customer Type,Type of Travel,Class,svc_sum,svc_mean
0,36,694,4,4,4,3,1,4,1,1,...,3,1,0,0.0,Female,Loyal Customer,Personal Travel,Eco,37,2.846154
1,56,651,5,5,5,5,5,4,4,5,...,4,3,0,0.0,Male,Loyal Customer,Business travel,Business,60,4.615385


In [17]:
# route stats: train + test pooled, pure feature statistics (no target anywhere)
tmp = pd.concat([X[['Flight Distance','Age','svc_sum']+svc],
                 X_test[['Flight Distance','Age','svc_sum']+svc]], ignore_index=True)
g = tmp.groupby('Flight Distance')

In [22]:
route = g.size().to_frame('route_count')

In [23]:
# in each of the variable, create new variable "mean" based on each group
route['route_age_mean'] = g['Age'].mean()
route['route_svcsum_mean'] = g['svc_sum'].mean()

In [27]:
# did the same thing as above with more automatic way
for c in svc:
    route['r_' + c[:12] + '_m'] = g[c].mean()

In [28]:
route = route.fillna(0)

X = X.join(route, on='Flight Distance')
X_test = X_test.join(route, on='Flight Distance')

## Basic Data Process for Modeling

Data preparation - Create Dummy variable - NO DUMMY

In [29]:
for df in [X, X_test]:
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype('category')

## Model using XGboost 5 fold CV

In [31]:
SEED = 42
N_FOLDS = 5
TARGET = 'satisfaction'

In [32]:
# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [33]:
def run_cv(fit_predict_fn, name):
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va)
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [34]:
final_params = {
    # fix parameter
        "objective":"binary:logistic",
        "eval_metric": "auc",
        "n_estimators": 5000,
        "tree_method": "hist",
        "enable_categorical": True,
        "random_state": 42,
        "colsample_bytree":0.8,
        "subsample":0.8,
        "n_jobs":-1,
        # early stopping round
        "early_stopping_rounds": 200,

        "learning_rate": 0.023621412607945612,
        "max_depth": 8,
        "reg_lambda": 0.15767822496360231,
        "alpha": 0.3574854835852679,
        "min_child_weight":29
}

In [35]:
def fit_xgb(X_tr, y_tr, X_va, y_va):
    model = xgb.XGBClassifier(**final_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [36]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, 'XGBoost')

  fold 0: AUC = 0.96072
  fold 1: AUC = 0.95953
  fold 2: AUC = 0.96118
  fold 3: AUC = 0.96064
  fold 4: AUC = 0.96039
XGBoost | OOF AUC = 0.96049 (mean 0.96049 ± 0.00055)


In [37]:
def save_sub(pred, name):
    sub = submit.copy()
    sub['satisfaction'] = pred
    sub.to_csv(name, index=False)
    print('saving file:', name)

In [38]:
save_sub(test_xgb,'sub_xgb_15_5cv_FE_mumu.csv')

saving file: sub_xgb_15_5cv_FE_mumu.csv


In [39]:
oof_xgb_df = train[['id']].copy()
oof_xgb_df.loc[:,'oof_xgb_15_5cv_fe_mumu'] = oof_xgb
oof_xgb_df.to_csv('oof_xgb_15_5cv_fe_mumu.csv')